In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

def complete_clustering_analysis(file_path):
    """Complete clustering analysis using ALL your columns"""
    
    print("🚀 COMPLETE FRIEND CLUSTERING ANALYSIS")
    print("=" * 50)
    
    # Load data
    df = pd.read_excel(file_path)
    print(f"✅ Loaded {len(df)} friends with {len(df.columns)} columns")
    
    # Columns to use for clustering (all except ID)
    feature_cols = [col for col in df.columns if col != 'Unique_ID']
    print(f"🎯 Using {len(feature_cols)} columns: {feature_cols}")
    
    # Prepare features
    encoders = {}
    processed_features = []
    feature_names = []
    
    for col in feature_cols:
        print(f"   Processing: {col}")
        
        # Handle missing values
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna('Unknown')
        
        # Encode categorical data
        encoders[col] = LabelEncoder()
        encoded_values = encoders[col].fit_transform(df[col].astype(str))
        processed_features.append(encoded_values)
        feature_names.append(f"{col}_encoded")
    
    # Combine features
    X = np.column_stack(processed_features)
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"✅ Feature preparation complete: {X_scaled.shape}")
    
    # Test different clustering configurations
    print(f"\n🎯 Testing optimal clustering configurations...")
    
    results = []
    methods = ['kmeans', 'gmm']
    k_range = range(3, 16)
    
    print(f"{'Method':<10} {'K':>3} {'Silhouette':>11} {'Quality':>10}")
    print("-" * 40)
    
    for method in methods:
        for k in k_range:
            try:
                if method == 'kmeans':
                    model = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
                    labels = model.fit_predict(X_scaled)
                elif method == 'gmm':
                    model = GaussianMixture(n_components=k, random_state=42)
                    model.fit(X_scaled)
                    labels = model.predict(X_scaled)
                
                # Calculate quality
                sil_score = silhouette_score(X_scaled, labels)
                quality = "Excellent" if sil_score > 0.5 else "Good" if sil_score > 0.3 else "Fair"
                
                results.append({
                    'method': method,
                    'k': k,
                    'silhouette': sil_score,
                    'model': model,
                    'labels': labels
                })
                
                print(f"{method.upper():<10} {k:>3} {sil_score:>11.3f} {quality:>10}")
                
            except Exception as e:
                continue
    
    # Get best configuration
    best_config = max(results, key=lambda x: x['silhouette'])
    best_labels = best_config['labels']
    
    print(f"\n🏆 BEST CONFIGURATION:")
    print(f"   Method: {best_config['method'].upper()}")
    print(f"   Clusters: {best_config['k']}")
    print(f"   Silhouette Score: {best_config['silhouette']:.3f}")
    
    # Add cluster assignments to dataframe
    df['cluster'] = best_labels
    
    # Analyze each cluster
    print(f"\n📊 CLUSTER ANALYSIS:")
    print("=" * 60)
    
    for cluster_id in sorted(df['cluster'].unique()):
        cluster_data = df[df['cluster'] == cluster_id]
        print(f"\n🏷️  Cluster {cluster_id} - {len(cluster_data)} friends ({len(cluster_data)/len(df)*100:.1f}%)")
        
        # Show characteristics for each original column
        for col in feature_cols:
            if col in cluster_data.columns:
                mode_val = cluster_data[col].mode()
                if len(mode_val) > 0:
                    print(f"   {col}: {mode_val.iloc[0]} (most common)")
        
        # Show sample friend IDs
        sample_ids = cluster_data['Unique_ID'].head(3).tolist()
        print(f"   Sample Friends: {sample_ids}")
    
    # Export results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f'complete_friend_clustering_{timestamp}.xlsx'
    
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        # All friends with cluster assignments
        df.to_excel(writer, sheet_name='Friends_with_Clusters', index=False)
        
        # Cluster summary
        cluster_summary = []
        for cluster_id in sorted(df['cluster'].unique()):
            cluster_data = df[df['cluster'] == cluster_id]
            
            summary = {
                'Cluster_ID': cluster_id,
                'Size': len(cluster_data),
                'Percentage': f"{len(cluster_data)/len(df)*100:.1f}%"
            }
            
            # Add most common values for each feature
            for col in feature_cols:
                mode_val = cluster_data[col].mode()
                summary[f'{col}_Most_Common'] = mode_val.iloc[0] if len(mode_val) > 0 else 'Mixed'
            
            cluster_summary.append(summary)
        
        summary_df = pd.DataFrame(cluster_summary)
        summary_df.to_excel(writer, sheet_name='Cluster_Summary', index=False)
    
    print(f"\n🎉 ANALYSIS COMPLETE!")
    print(f"✅ {len(df)} friends clustered into {best_config['k']} groups")
    print(f"📁 Results saved to: {output_filename}")
    
    return df, output_filename

# Run the complete analysis
df_with_clusters, results_file = complete_clustering_analysis('./las3.18.xlsx')

# Show sample results
print(f"\n📋 SAMPLE RESULTS:")
sample = df_with_clusters[['Unique_ID', 'Parent_Industry', 'Job_Function', 'cluster']].head(10)
print(sample.to_string(index=False))


🚀 COMPLETE FRIEND CLUSTERING ANALYSIS
✅ Loaded 2425 friends with 11 columns
🎯 Using 10 columns: ['Parent_Industry', 'Turnover_Range', 'Employees_Range', 'Job_Seniority', 'Job_Function', 'aoi_1', 'aoi_2', 'aoi_3', 'aoi_4', 'aoi_5']
   Processing: Parent_Industry
   Processing: Turnover_Range
   Processing: Employees_Range
   Processing: Job_Seniority
   Processing: Job_Function
   Processing: aoi_1
   Processing: aoi_2
   Processing: aoi_3
   Processing: aoi_4
   Processing: aoi_5
✅ Feature preparation complete: (2425, 10)

🎯 Testing optimal clustering configurations...
Method       K  Silhouette    Quality
----------------------------------------
KMEANS       3       0.116       Fair
KMEANS       4       0.117       Fair
KMEANS       5       0.110       Fair
KMEANS       6       0.107       Fair
KMEANS       7       0.107       Fair
KMEANS       8       0.108       Fair
KMEANS       9       0.104       Fair
KMEANS      10       0.109       Fair
KMEANS      11       0.110       Fair
KME

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from datetime import datetime
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

def gmm_clustering_analysis(file_path):
    """Complete clustering analysis using GMM specifically"""
    
    print("🚀 GMM CLUSTERING ANALYSIS FOR FRIENDS")
    print("=" * 50)
    
    # Load data
    df = pd.read_excel(file_path)
    print(f"✅ Loaded {len(df)} friends with {len(df.columns)} columns")
    
    # Columns to use for clustering (all except ID)
    feature_cols = [col for col in df.columns if col != 'Unique_ID']
    print(f"🎯 Using {len(feature_cols)} columns: {feature_cols}")
    
    # Prepare features
    encoders = {}
    processed_features = []
    feature_names = []
    
    for col in feature_cols:
        print(f"   Processing: {col}")
        
        # Handle missing values
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna('Unknown')
        
        # Encode categorical data
        encoders[col] = LabelEncoder()
        encoded_values = encoders[col].fit_transform(df[col].astype(str))
        processed_features.append(encoded_values)
        feature_names.append(f"{col}_encoded")
    
    # Combine and scale features
    X = np.column_stack(processed_features)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"✅ Feature preparation complete: {X_scaled.shape}")
    
    # Find optimal GMM configuration
    print(f"\n🎯 Finding optimal GMM clusters using BIC/AIC...")
    
    max_clusters = 20
    gmm_results = []
    
    print(f"{'K':>3} {'BIC':>10} {'AIC':>10} {'Silhouette':>11} {'Quality':>10}")
    print("-" * 50)
    
    for k in range(2, max_clusters + 1):
        try:
            # Fit GMM
            gmm = GaussianMixture(
                n_components=k,
                covariance_type='full',  # Most flexible
                init_params='kmeans',    # Better initialization
                max_iter=200,
                random_state=42
            )
            
            gmm.fit(X_scaled)
            
            # Calculate metrics
            bic = gmm.bic(X_scaled)
            aic = gmm.aic(X_scaled)
            labels = gmm.predict(X_scaled)
            sil_score = silhouette_score(X_scaled, labels)
            
            # Quality assessment
            if sil_score > 0.5:
                quality = "Excellent"
            elif sil_score > 0.3:
                quality = "Good"
            elif sil_score > 0.2:
                quality = "Fair"
            else:
                quality = "Poor"
            
            gmm_results.append({
                'k': k,
                'bic': bic,
                'aic': aic,
                'silhouette': sil_score,
                'quality': quality,
                'model': gmm,
                'labels': labels
            })
            
            print(f"{k:>3} {bic:>10.1f} {aic:>10.1f} {sil_score:>11.3f} {quality:>10}")
            
        except Exception as e:
            continue
    
    # Select best model using BIC (lower is better)
    best_gmm = min(gmm_results, key=lambda x: x['bic'])
    
    print(f"\n🏆 OPTIMAL GMM CONFIGURATION:")
    print(f"   Components: {best_gmm['k']}")
    print(f"   BIC Score: {best_gmm['bic']:.1f} (lower is better)")
    print(f"   AIC Score: {best_gmm['aic']:.1f}")
    print(f"   Silhouette Score: {best_gmm['silhouette']:.3f}")
    print(f"   Quality: {best_gmm['quality']}")
    
    # Get final cluster assignments
    final_labels = best_gmm['labels']
    df['cluster'] = final_labels
    
    # Get cluster probabilities (soft clustering)
    cluster_probabilities = best_gmm['model'].predict_proba(X_scaled)
    
    # Add probability information
    for i in range(best_gmm['k']):
        df[f'cluster_{i}_probability'] = cluster_probabilities[:, i]
    
    # Analyze each cluster
    print(f"\n📊 GMM CLUSTER ANALYSIS:")
    print("=" * 60)
    
    for cluster_id in sorted(df['cluster'].unique()):
        cluster_data = df[df['cluster'] == cluster_id]
        print(f"\n🏷️  Cluster {cluster_id} - {len(cluster_data)} friends ({len(cluster_data)/len(df)*100:.1f}%)")
        
        # Show characteristics for each original column
        for col in feature_cols:
            if col in cluster_data.columns:
                mode_val = cluster_data[col].mode()
                if len(mode_val) > 0:
                    print(f"   {col}: {mode_val.iloc[0]} (most common)")
        
        # Show sample friend IDs with their cluster confidence
        sample_friends = cluster_data.head(3)
        print(f"   Sample Friends:")
        for _, friend in sample_friends.iterrows():
            friend_id = friend['Unique_ID']
            confidence = friend[f'cluster_{cluster_id}_probability']
            print(f"     {friend_id} (confidence: {confidence:.3f})")
    
    # Analyze cluster overlap (unique to GMM)
    print(f"\n🔄 CLUSTER OVERLAP ANALYSIS:")
    print("=" * 40)
    
    # Find friends with high uncertainty (belong to multiple clusters)
    max_probs = cluster_probabilities.max(axis=1)
    uncertain_friends = df[max_probs < 0.7]  # Less than 70% certainty
    
    print(f"Friends with high uncertainty: {len(uncertain_friends)} ({len(uncertain_friends)/len(df)*100:.1f}%)")
    
    if len(uncertain_friends) > 0:
        print("Sample uncertain friends:")
        for _, friend in uncertain_friends.head(3).iterrows():
            friend_id = friend['Unique_ID']
            probs = [friend[f'cluster_{i}_probability'] for i in range(best_gmm['k'])]
            top_clusters = sorted(enumerate(probs), key=lambda x: x[1], reverse=True)[:2]
            print(f"   {friend_id}: Cluster {top_clusters[0][0]} ({top_clusters[0][1]:.3f}) | Cluster {top_clusters[1][0]} ({top_clusters[1][1]:.3f})")
    
    # Export results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f'gmm_friend_clustering_{timestamp}.xlsx'
    
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        # All friends with cluster assignments and probabilities
        df.to_excel(writer, sheet_name='Friends_with_GMM_Clusters', index=False)
        
        # Cluster summary
        cluster_summary = []
        for cluster_id in sorted(df['cluster'].unique()):
            cluster_data = df[df['cluster'] == cluster_id]
            
            summary = {
                'Cluster_ID': cluster_id,
                'Size': len(cluster_data),
                'Percentage': f"{len(cluster_data)/len(df)*100:.1f}%",
                'Avg_Confidence': cluster_data[f'cluster_{cluster_id}_probability'].mean()
            }
            
            # Add most common values for each feature
            for col in feature_cols:
                mode_val = cluster_data[col].mode()
                summary[f'{col}_Most_Common'] = mode_val.iloc[0] if len(mode_val) > 0 else 'Mixed'
            
            cluster_summary.append(summary)
        
        summary_df = pd.DataFrame(cluster_summary)
        summary_df.to_excel(writer, sheet_name='GMM_Cluster_Summary', index=False)
        
        # Uncertain friends analysis
        if len(uncertain_friends) > 0:
            uncertain_analysis = []
            for _, friend in uncertain_friends.iterrows():
                friend_data = {'Unique_ID': friend['Unique_ID']}
                for i in range(best_gmm['k']):
                    friend_data[f'Cluster_{i}_Prob'] = friend[f'cluster_{i}_probability']
                uncertain_analysis.append(friend_data)
            
            uncertain_df = pd.DataFrame(uncertain_analysis)
            uncertain_df.to_excel(writer, sheet_name='Uncertain_Friends', index=False)
        
        # GMM model parameters
        model_info = [{
            'Parameter': 'Optimal_Components',
            'Value': best_gmm['k']
        }, {
            'Parameter': 'BIC_Score',
            'Value': best_gmm['bic']
        }, {
            'Parameter': 'AIC_Score', 
            'Value': best_gmm['aic']
        }, {
            'Parameter': 'Silhouette_Score',
            'Value': best_gmm['silhouette']
        }, {
            'Parameter': 'Covariance_Type',
            'Value': 'full'
        }]
        
        model_df = pd.DataFrame(model_info)
        model_df.to_excel(writer, sheet_name='GMM_Model_Info', index=False)
    
    print(f"\n🎉 GMM ANALYSIS COMPLETE!")
    print(f"✅ {len(df)} friends clustered into {best_gmm['k']} GMM components")
    print(f"📁 Results saved to: {output_filename}")
    print(f"🔄 Includes soft clustering probabilities and uncertainty analysis")
    
    return df, best_gmm['model'], output_filename

# Alternative: Bayesian GMM for automatic cluster number inference
def bayesian_gmm_clustering(file_path, max_components=20):
    """Use Bayesian GMM to automatically infer number of clusters"""
    
    print("🚀 BAYESIAN GMM CLUSTERING (AUTO CLUSTER DETECTION)")
    print("=" * 60)
    
    # Load and prepare data (same as above)
    df = pd.read_excel(file_path)
    feature_cols = [col for col in df.columns if col != 'Unique_ID']
    
    # Encode features
    processed_features = []
    for col in feature_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna('Unknown')
        
        le = LabelEncoder()
        encoded_values = le.fit_transform(df[col].astype(str))
        processed_features.append(encoded_values)
    
    X = np.column_stack(processed_features)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Bayesian GMM - automatically determines effective number of components
    bgmm = BayesianGaussianMixture(
        n_components=max_components,  # Upper bound
        weight_concentration_prior=0.1,  # Lower = fewer components
        covariance_type='full',
        init_params='kmeans',
        max_iter=300,
        random_state=42
    )
    
    bgmm.fit(X_scaled)
    labels = bgmm.predict(X_scaled)
    
    # Check effective number of components
    weights = bgmm.weights_
    active_components = np.sum(weights > 0.01)
    effective_clusters = len(np.unique(labels))
    
    print(f"🎯 Bayesian GMM Results:")
    print(f"   Maximum components: {max_components}")
    print(f"   Active components: {active_components}")
    print(f"   Effective clusters: {effective_clusters}")
    print(f"   Component weights: {weights[weights > 0.01]}")
    
    df['cluster'] = labels
    
    return df, bgmm

# Run GMM clustering
print("Choose GMM method:")
print("1. Standard GMM with BIC/AIC optimization")
print("2. Bayesian GMM with automatic detection")

# For standard GMM (recommended)
df_with_clusters, gmm_model, results_file = gmm_clustering_analysis('./las3.18.xlsx')

# Show sample results
print(f"\n📋 SAMPLE GMM RESULTS:")
sample = df_with_clusters[['Unique_ID', 'Parent_Industry', 'Job_Function', 'cluster', 
                          'cluster_0_probability', 'cluster_1_probability']].head(5)
print(sample.to_string(index=False))


Choose GMM method:
1. Standard GMM with BIC/AIC optimization
2. Bayesian GMM with automatic detection
🚀 GMM CLUSTERING ANALYSIS FOR FRIENDS
✅ Loaded 2425 friends with 11 columns
🎯 Using 10 columns: ['Parent_Industry', 'Turnover_Range', 'Employees_Range', 'Job_Seniority', 'Job_Function', 'aoi_1', 'aoi_2', 'aoi_3', 'aoi_4', 'aoi_5']
   Processing: Parent_Industry
   Processing: Turnover_Range
   Processing: Employees_Range
   Processing: Job_Seniority
   Processing: Job_Function
   Processing: aoi_1
   Processing: aoi_2
   Processing: aoi_3
   Processing: aoi_4
   Processing: aoi_5
✅ Feature preparation complete: (2425, 10)

🎯 Finding optimal GMM clusters using BIC/AIC...
  K        BIC        AIC  Silhouette    Quality
--------------------------------------------------
  2    49310.3    48551.4       0.082       Poor
  3    48780.5    47639.2      -0.002       Poor
  4    44659.3    43135.6       0.011       Poor
  5    43907.6    42001.5       0.031       Poor
  6    41540.1    39251.6

In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

def complete_clustering_analysis_all_columns(file_path):
    """Complete clustering analysis using ALL columns automatically"""
    
    print("🚀 CLUSTERING ANALYSIS USING ALL COLUMNS")
    print("=" * 50)
    
    # Load data
    df = pd.read_excel(file_path)
    print(f"✅ Loaded {len(df)} friends with {len(df.columns)} columns")
    
    # Automatically use ALL columns (exclude ID for clustering, but keep it in results)
    all_columns = list(df.columns)
    id_column = 'Unique_ID'  # Assuming this is your ID column
    feature_cols = [col for col in all_columns if col != id_column]
    
    print(f"🎯 Using ALL {len(feature_cols)} columns for clustering: {feature_cols}")
    print(f"🆔 ID column '{id_column}' preserved in results")
    
    # Prepare features
    encoders = {}
    processed_features = []
    feature_names = []
    
    for col in feature_cols:
        print(f"   Processing: {col}")
        
        # Handle missing values
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna('Unknown')
        
        # Encode categorical data
        encoders[col] = LabelEncoder()
        encoded_values = encoders[col].fit_transform(df[col].astype(str))
        processed_features.append(encoded_values)
        feature_names.append(f"{col}_encoded")
    
    # Combine and scale features
    X = np.column_stack(processed_features)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"✅ Feature preparation complete: {X_scaled.shape}")
    
    # Find optimal configuration
    print(f"\n🎯 Finding optimal clustering configuration...")
    
    results = []
    methods = ['kmeans', 'gmm']
    k_range = range(3, 16)
    
    print(f"{'Method':<10} {'K':>3} {'Silhouette':>11} {'Quality':>10}")
    print("-" * 40)
    
    for method in methods:
        for k in k_range:
            try:
                # Fit model
                if method == 'kmeans':
                    model = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
                    labels = model.fit_predict(X_scaled)
                elif method == 'gmm':
                    model = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
                    model.fit(X_scaled)
                    labels = model.predict(X_scaled)
                
                # Calculate quality
                sil_score = silhouette_score(X_scaled, labels)
                quality = "Excellent" if sil_score > 0.5 else "Good" if sil_score > 0.3 else "Fair"
                
                results.append({
                    'method': method,
                    'k': k,
                    'silhouette': sil_score,
                    'model': model,
                    'labels': labels
                })
                
                print(f"{method.upper():<10} {k:>3} {sil_score:>11.3f} {quality:>10}")
                
            except Exception as e:
                continue
    
    # Get best configuration
    best_config = max(results, key=lambda x: x['silhouette'])
    best_labels = best_config['labels']
    
    print(f"\n🏆 BEST CONFIGURATION:")
    print(f"   Method: {best_config['method'].upper()}")
    print(f"   Clusters: {best_config['k']}")
    print(f"   Silhouette Score: {best_config['silhouette']:.3f}")
    
    # Add cluster assignments to dataframe
    df['cluster'] = best_labels
    
    # Analyze each cluster
    print(f"\n📊 CLUSTER ANALYSIS:")
    print("=" * 60)
    
    for cluster_id in sorted(df['cluster'].unique()):
        cluster_data = df[df['cluster'] == cluster_id]
        print(f"\n🏷️  Cluster {cluster_id} - {len(cluster_data)} friends ({len(cluster_data)/len(df)*100:.1f}%)")
        
        # Show characteristics for ALL original columns
        for col in feature_cols:
            if col in cluster_data.columns:
                mode_val = cluster_data[col].mode()
                if len(mode_val) > 0:
                    print(f"   {col}: {mode_val.iloc[0]} (most common)")
        
        # Show sample friend IDs
        sample_friends = cluster_data.head(3)
        print(f"   Sample Friends:")
        for _, friend in sample_friends.iterrows():
            friend_id = friend['Unique_ID']
            print(f"     {friend_id}")
    
    # Export results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f'complete_friend_clustering_{timestamp}.xlsx'
    
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        # All friends with cluster assignments
        df.to_excel(writer, sheet_name='Friends_with_Clusters', index=False)
        
        # Cluster summary
        cluster_summary = []
        for cluster_id in sorted(df['cluster'].unique()):
            cluster_data = df[df['cluster'] == cluster_id]
            
            summary = {
                'Cluster_ID': cluster_id,
                'Size': len(cluster_data),
                'Percentage': f"{len(cluster_data)/len(df)*100:.1f}%"
            }
            
            # Add most common values for each feature
            for col in feature_cols:
                mode_val = cluster_data[col].mode()
                summary[f'{col}_Most_Common'] = mode_val.iloc[0] if len(mode_val) > 0 else 'Mixed'
            
            cluster_summary.append(summary)
        
        summary_df = pd.DataFrame(cluster_summary)
        summary_df.to_excel(writer, sheet_name='Cluster_Summary', index=False)
    
    print(f"\n🎉 ANALYSIS COMPLETE!")
    print(f"✅ {len(df)} friends clustered into {best_config['k']} groups")
    print(f"📁 Results saved to: {output_filename}")
    
    return df, output_filename

# Run the complete analysis
df_with_clusters, results_file = complete_clustering_analysis_all_columns('./las3.18.xlsx')


🚀 CLUSTERING ANALYSIS USING ALL COLUMNS
✅ Loaded 2425 friends with 11 columns
🎯 Using ALL 10 columns for clustering: ['Parent_Industry', 'Turnover_Range', 'Employees_Range', 'Job_Seniority', 'Job_Function', 'aoi_1', 'aoi_2', 'aoi_3', 'aoi_4', 'aoi_5']
🆔 ID column 'Unique_ID' preserved in results
   Processing: Parent_Industry
   Processing: Turnover_Range
   Processing: Employees_Range
   Processing: Job_Seniority
   Processing: Job_Function
   Processing: aoi_1
   Processing: aoi_2
   Processing: aoi_3
   Processing: aoi_4
   Processing: aoi_5
✅ Feature preparation complete: (2425, 10)

🎯 Finding optimal clustering configuration...
Method       K  Silhouette    Quality
----------------------------------------
KMEANS       3       0.116       Fair
KMEANS       4       0.117       Fair
KMEANS       5       0.110       Fair
KMEANS       6       0.107       Fair
KMEANS       7       0.107       Fair
KMEANS       8       0.108       Fair
KMEANS       9       0.104       Fair
KMEANS      10

In [ ]:


import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA, FastICA, FactorAnalysis
from sklearn.manifold import TSNE
from umap import UMAP
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

class AdvancedGMMClusteringSystem:
    """
    Advanced GMM clustering system designed to discover unique and meaningful clusters
    """
    
    def __init__(self):
        self.scalers = {}
        self.models = {}
        self.results = []
        self.best_configuration = None
        self.processed_data = None
        
    def advanced_data_preprocessing(self, df, feature_cols, preprocessing_strategies=None):
        """Advanced preprocessing with multiple strategies"""
        
        if preprocessing_strategies is None:
            preprocessing_strategies = [
                'robust_encoding',
                'embedding_enhanced', 
                'feature_engineered',
                'outlier_removed',
                'variance_filtered'
            ]
        
        print("🔧 Advanced data preprocessing...")
        
        preprocessed_data = {}
        
        for strategy in preprocessing_strategies:
            if strategy == 'robust_encoding':
                preprocessed_data[strategy] = self._robust_encoding_preprocessing(df, feature_cols)
            elif strategy == 'embedding_enhanced':
                preprocessed_data[strategy] = self._embedding_enhanced_preprocessing(df)
            elif strategy == 'feature_engineered':
                preprocessed_data[strategy] = self._feature_engineering_preprocessing(df, feature_cols)
            elif strategy == 'outlier_removed':
                preprocessed_data[strategy] = self._outlier_removal_preprocessing(df, feature_cols)
            elif strategy == 'variance_filtered':
                preprocessed_data[strategy] = self._variance_filtering_preprocessing(df, feature_cols)
        
        return preprocessed_data
    
    def _robust_encoding_preprocessing(self, df, feature_cols):
        """Robust encoding with better categorical handling"""
        
        processed_df = df.copy()
        
        # Advanced categorical encoding
        for col in feature_cols:
            if processed_df[col].dtype == 'object':
                # Create frequency-based encoding for better separation
                freq_map = processed_df[col].value_counts().to_dict()
                processed_df[col + '_freq'] = processed_df[col].map(freq_map)
                
                # Standard factorization
                processed_df[col + '_encoded'] = pd.factorize(processed_df[col].fillna('Unknown'))[0]
                
                # Drop original column
                processed_df = processed_df.drop(columns=[col])
            else:
                processed_df[col] = processed_df[col].fillna(processed_df[col].median())
        
        # Use RobustScaler instead of StandardScaler for better outlier handling
        scaler = RobustScaler()
        X_scaled = scaler.fit_transform(processed_df[processed_df.select_dtypes(include=[np.number]).columns])
        
        self.scalers['robust_encoding'] = scaler
        print(f"   ✅ Robust encoding: {X_scaled.shape}")
        
        return X_scaled
    
    def _embedding_enhanced_preprocessing(self, df):
        """Enhanced embedding with multiple semantic representations"""
        
        # Create multiple text representations for richer embeddings
        text_variations = []
        
        for _, row in df.iterrows():
            # Variation 1: Professional profile
            prof_text = f"Senior professional in {row['Parent_Industry']} industry, {row['Job_Function']} role at {row['Job_Seniority']} level"
            
            # Variation 2: Company profile  
            comp_text = f"Works at company with {row['Turnover_Range']} revenue and {row['Employees_Range']} employees"
            
            # Variation 3: Interest profile
            interest_text = f"Interested in {row['aoi_1']}, {row['aoi_2']}, {row['aoi_3']}, {row['aoi_4']}, {row['aoi_5']}"
            
            # Combined representation
            combined_text = f"{prof_text}. {comp_text}. {interest_text}"
            text_variations.append(combined_text)
        
        # Generate embeddings
        model = SentenceTransformer('all-MiniLM-L6-v2')
        embeddings = model.encode(text_variations, show_progress_bar=True)
        
        # Apply additional dimensionality reduction for better clustering
        pca = PCA(n_components=50, random_state=42)
        embeddings_reduced = pca.fit_transform(embeddings)
        
        # Scale embeddings
        scaler = StandardScaler()
        embeddings_scaled = scaler.fit_transform(embeddings_reduced)
        
        self.scalers['embedding_enhanced'] = scaler
        print(f"   ✅ Enhanced embeddings: {embeddings_scaled.shape}")
        
        return embeddings_scaled
    
    def _feature_engineering_preprocessing(self, df, feature_cols):
        """Advanced feature engineering for better cluster discovery"""
        
        processed_df = df.copy()
        
        # Encode categorical variables first
        encoded_cols = []
        for col in feature_cols:
            if processed_df[col].dtype == 'object':
                processed_df[col + '_encoded'] = pd.factorize(processed_df[col].fillna('Unknown'))[0]
                encoded_cols.append(col + '_encoded')
            else:
                processed_df[col] = processed_df[col].fillna(processed_df[col].median())
                encoded_cols.append(col)
        
        # Feature engineering
        feature_matrix = processed_df[encoded_cols].values
        
        # Create interaction features (combinations that might reveal hidden patterns)
        if len(encoded_cols) >= 2:
            # Industry-Function interaction
            if 'Parent_Industry_encoded' in encoded_cols and 'Job_Function_encoded' in encoded_cols:
                industry_idx = encoded_cols.index('Parent_Industry_encoded')
                function_idx = encoded_cols.index('Job_Function_encoded')
                interaction = feature_matrix[:, industry_idx] * feature_matrix[:, function_idx]
                feature_matrix = np.column_stack([feature_matrix, interaction])
            
            # Seniority-Company size interaction
            if 'Job_Seniority_encoded' in encoded_cols and 'Employees_Range_encoded' in encoded_cols:
                seniority_idx = encoded_cols.index('Job_Seniority_encoded')
                company_idx = encoded_cols.index('Employees_Range_encoded')
                interaction = feature_matrix[:, seniority_idx] * feature_matrix[:, company_idx]
                feature_matrix = np.column_stack([feature_matrix, interaction])
        
        # Apply multiple transformations
        # 1. Original scaled features
        scaler1 = StandardScaler()
        scaled_features = scaler1.fit_transform(feature_matrix)
        
        # 2. PCA features for linear relationships
        pca = PCA(n_components=min(20, feature_matrix.shape[1]), random_state=42)
        pca_features = pca.fit_transform(scaled_features)
        
        # 3. ICA features for non-linear relationships
        ica = FastICA(n_components=min(15, feature_matrix.shape[1]), random_state=42)
        ica_features = ica.fit_transform(scaled_features)
        
        # Combine all features
        combined_features = np.hstack([scaled_features, pca_features, ica_features])
        
        # Final scaling
        final_scaler = StandardScaler()
        final_features = final_scaler.fit_transform(combined_features)
        
        self.scalers['feature_engineered'] = final_scaler
        print(f"   ✅ Feature engineered: {final_features.shape}")
        
        return final_features
    
    def _outlier_removal_preprocessing(self, df, feature_cols):
        """Remove outliers to improve clustering quality"""
        
        # First do standard preprocessing
        processed_df = df.copy()
        for col in feature_cols:
            if processed_df[col].dtype == 'object':
                processed_df[col] = pd.factorize(processed_df[col].fillna('Unknown'))[0]
            else:
                processed_df[col] = processed_df[col].fillna(processed_df[col].median())
        
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(processed_df[feature_cols])
        
        # Detect outliers using multiple methods
        iso_forest = IsolationForest(contamination=0.1, random_state=42)
        outliers_iso = iso_forest.fit_predict(X_scaled)
        
        lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
        outliers_lof = lof.fit_predict(X_scaled)
        
        # Keep only inliers (both methods agree)
        inliers_mask = (outliers_iso == 1) & (outliers_lof == 1)
        X_cleaned = X_scaled[inliers_mask]
        
        self.scalers['outlier_removed'] = scaler
        print(f"   ✅ Outlier removed: {X_cleaned.shape} (removed {np.sum(~inliers_mask)} outliers)")
        
        return X_cleaned
    
    def _variance_filtering_preprocessing(self, df, feature_cols):
        """Filter low-variance features to focus on discriminative ones"""
        
        # Standard preprocessing
        processed_df = df.copy()
        for col in feature_cols:
            if processed_df[col].dtype == 'object':
                processed_df[col] = pd.factorize(processed_df[col].fillna('Unknown'))[0]
            else:
                processed_df[col] = processed_df[col].fillna(processed_df[col].median())
        
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(processed_df[feature_cols])
        
        # Remove low-variance features
        variance_filter = VarianceThreshold(threshold=0.1)
        X_filtered = variance_filter.fit_transform(X_scaled)
        
        self.scalers['variance_filtered'] = scaler
        print(f"   ✅ Variance filtered: {X_filtered.shape}")
        
        return X_filtered
    
    def advanced_gmm_optimization(self, data_dict, advanced_configs=None):
        """Advanced GMM optimization with multiple strategies"""
        
        if advanced_configs is None:
            advanced_configs = {
                'standard_gmm': {
                    'covariance_types': ['full', 'tied', 'diag'],
                    'k_range': range(3, 20),
                    'init_params': ['kmeans', 'random']
                },
                'bayesian_gmm': {
                    'k_range': range(3, 25),
                    'weight_concentration_priors': [0.01, 0.1, 1.0]
                },
                'ensemble_gmm': {
                    'n_estimators': 5,
                    'k_range': range(3, 15)
                }
            }
        
        print(f"\n🎯 Advanced GMM optimization on {len(data_dict)} data representations...")
        print(f"{'Data Type':<20} {'GMM Type':<15} {'Config':>12} {'K':>3} {'Silhouette':>11} {'Calinski':>11} {'Davies':>10} {'Quality':>12}")
        print("-" * 110)
        
        for data_name, X in data_dict.items():
            print(f"\n--- Processing {data_name} ---")
            
            # Standard GMM with multiple configurations
            self._optimize_standard_gmm(data_name, X, advanced_configs['standard_gmm'])
            
            # Bayesian GMM for automatic component selection
            self._optimize_bayesian_gmm(data_name, X, advanced_configs['bayesian_gmm'])
            
            # Ensemble GMM for robustness
            self._optimize_ensemble_gmm(data_name, X, advanced_configs['ensemble_gmm'])
        
        return self.results
    
    def _optimize_standard_gmm(self, data_name, X, config):
        """Optimize standard GMM with multiple configurations"""
        
        for cov_type in config['covariance_types']:
            for init_param in config['init_params']:
                for k in config['k_range']:
                    try:
                        # Multiple random initializations for robustness
                        best_score = -1
                        best_model = None
                        best_labels = None
                        
                        for random_state in [42, 123, 456, 789, 999]:
                            gmm = GaussianMixture(
                                n_components=k,
                                covariance_type=cov_type,
                                init_params=init_param,
                                n_init=10,
                                max_iter=200,
                                random_state=random_state
                            )
                            
                            gmm.fit(X)
                            labels = gmm.predict(X)
                            
                            if len(set(labels)) >= 2:
                                sil_score = silhouette_score(X, labels)
                                if sil_score > best_score:
                                    best_score = sil_score
                                    best_model = gmm
                                    best_labels = labels
                        
                        if best_model is not None:
                            cal_score = calinski_harabasz_score(X, best_labels)
                            dbi_score = davies_bouldin_score(X, best_labels)
                            
                            # Enhanced quality assessment
                            if best_score > 0.7:
                                quality = "Excellent"
                            elif best_score > 0.5:
                                quality = "Very Good"
                            elif best_score > 0.3:
                                quality = "Good"
                            elif best_score > 0.2:
                                quality = "Fair"
                            else:
                                quality = "Poor"
                            
                            result = {
                                'data_type': data_name,
                                'gmm_type': 'standard',
                                'config': f'{cov_type}-{init_param}',
                                'k': k,
                                'silhouette': best_score,
                                'calinski': cal_score,
                                'davies': dbi_score,
                                'quality': quality,
                                'model': best_model,
                                'labels': best_labels,
                                'converged': best_model.converged_,
                                'n_iter': best_model.n_iter_
                            }
                            
                            self.results.append(result)
                            
                            print(f"{data_name:<20} {'Standard':<15} {f'{cov_type}-{init_param}':>12} {k:>3} {best_score:>11.3f} {cal_score:>11.2f} {dbi_score:>10.3f} {quality:>12}")
                    
                    except Exception as e:
                        continue
    
    def _optimize_bayesian_gmm(self, data_name, X, config):
        """Optimize Bayesian GMM for automatic component selection"""
        
        for weight_prior in config['weight_concentration_priors']:
            for max_k in config['k_range']:
                try:
                    bgmm = BayesianGaussianMixture(
                        n_components=max_k,
                        weight_concentration_prior=weight_prior,
                        covariance_type='full',
                        max_iter=300,
                        random_state=42
                    )
                    
                    bgmm.fit(X)
                    labels = bgmm.predict(X)
                    
                    # Check effective number of components
                    active_components = np.sum(bgmm.weights_ > 0.01)
                    unique_labels = len(set(labels))
                    
                    if unique_labels >= 2:
                        sil_score = silhouette_score(X, labels)
                        cal_score = calinski_harabasz_score(X, labels)
                        dbi_score = davies_bouldin_score(X, labels)
                        
                        quality = "Excellent" if sil_score > 0.7 else "Very Good" if sil_score > 0.5 else "Good" if sil_score > 0.3 else "Fair"
                        
                        result = {
                            'data_type': data_name,
                            'gmm_type': 'bayesian',
                            'config': f'prior={weight_prior}',
                            'k': unique_labels,
                            'silhouette': sil_score,
                            'calinski': cal_score,
                            'davies': dbi_score,
                            'quality': quality,
                            'model': bgmm,
                            'labels': labels,
                            'active_components': active_components,
                            'converged': bgmm.converged_
                        }
                        
                        self.results.append(result)
                        
                        print(f"{data_name:<20} {'Bayesian':<15} {f'prior={weight_prior}':>12} {unique_labels:>3} {sil_score:>11.3f} {cal_score:>11.2f} {dbi_score:>10.3f} {quality:>12}")
                
                except Exception as e:
                    continue
    
    def _optimize_ensemble_gmm(self, data_name, X, config):
        """Optimize ensemble GMM for robust clustering"""
        
        for k in config['k_range']:
            try:
                # Create ensemble of GMM models
                ensemble_models = []
                ensemble_labels = []
                
                for i in range(config['n_estimators']):
                    gmm = GaussianMixture(
                        n_components=k,
                        covariance_type='full',
                        n_init=5,
                        random_state=42 + i * 100
                    )
                    
                    gmm.fit(X)
                    labels = gmm.predict(X)
                    ensemble_models.append(gmm)
                    ensemble_labels.append(labels)
                
                # Consensus clustering (majority vote)
                ensemble_labels = np.array(ensemble_labels)
                consensus_labels = []
                
                for i in range(X.shape[0]):
                    votes = ensemble_labels[:, i]
                    consensus_labels.append(np.bincount(votes).argmax())
                
                consensus_labels = np.array(consensus_labels)
                
                if len(set(consensus_labels)) >= 2:
                    sil_score = silhouette_score(X, consensus_labels)
                    cal_score = calinski_harabasz_score(X, consensus_labels)
                    dbi_score = davies_bouldin_score(X, consensus_labels)
                    
                    quality = "Excellent" if sil_score > 0.7 else "Very Good" if sil_score > 0.5 else "Good" if sil_score > 0.3 else "Fair"
                    
                    result = {
                        'data_type': data_name,
                        'gmm_type': 'ensemble',
                        'config': f'n={config["n_estimators"]}',
                        'k': k,
                        'silhouette': sil_score,
                        'calinski': cal_score,
                        'davies': dbi_score,
                        'quality': quality,
                        'model': ensemble_models,  # Store all models
                        'labels': consensus_labels,
                        'ensemble_agreement': self._calculate_ensemble_agreement(ensemble_labels)
                    }
                    
                    self.results.append(result)
                    
                    print(f"{data_name:<20} {'Ensemble':<15} {'n=' + str(config['n_estimators']):>12} {k:>3} {sil_score:>11.3f} {cal_score:>11.2f} {dbi_score:>10.3f} {quality:>12}")

            
            except Exception as e:
                continue
    
    def _calculate_ensemble_agreement(self, ensemble_labels):
        """Calculate agreement between ensemble models"""
        n_models, n_samples = ensemble_labels.shape
        agreements = []
        
        for i in range(n_samples):
            votes = ensemble_labels[:, i]
            max_votes = np.bincount(votes).max()
            agreement = max_votes / n_models
            agreements.append(agreement)
        
        return np.mean(agreements)
    
    def find_best_unique_clusters(self):
        """Find configurations that produce the most unique and well-separated clusters"""
        
        if not self.results:
            print("❌ No results to analyze")
            return None
        
        print(f"\n🏆 FINDING BEST UNIQUE CLUSTERING CONFIGURATIONS...")
        
        # Convert to DataFrame for analysis
        results_df = pd.DataFrame([{k: v for k, v in result.items() if k not in ['model', 'labels']} 
                                  for result in self.results])
        
        # Multiple criteria for "uniqueness"
        # 1. High silhouette score (good separation)
        # 2. Reasonable number of clusters (not too few, not too many)
        # 3. Balanced cluster sizes
        # 4. High Calinski-Harabasz score
        # 5. Low Davies-Bouldin score
        
        # Calculate composite uniqueness score
        results_df['uniqueness_score'] = (
            0.4 * results_df['silhouette'] +  # Separation quality
            0.2 * (results_df['calinski'] / results_df['calinski'].max()) +  # Cluster definition
            0.2 * (1 - results_df['davies'] / results_df['davies'].max()) +  # Compactness
            0.2 * np.where((results_df['k'] >= 4) & (results_df['k'] <= 12), 1, 0.5)  # Reasonable K
        )
        
        # Find best configurations
        best_overall = results_df.loc[results_df['uniqueness_score'].idxmax()]
        best_silhouette = results_df.loc[results_df['silhouette'].idxmax()]
        best_balanced = results_df[(results_df['k'] >= 5) & (results_df['k'] <= 10)].loc[
            results_df[(results_df['k'] >= 5) & (results_df['k'] <= 10)]['silhouette'].idxmax()
        ] if len(results_df[(results_df['k'] >= 5) & (results_df['k'] <= 10)]) > 0 else best_overall
        
        # Store best configuration
        best_result_idx = results_df['uniqueness_score'].idxmax()
        self.best_configuration = self.results[best_result_idx]
        
        print(f"\n🥇 BEST UNIQUE CLUSTERING CONFIGURATION:")
        print(f"   Data Type: {best_overall['data_type']}")
        print(f"   GMM Type: {best_overall['gmm_type']}")
        print(f"   Config: {best_overall['config']}")
        print(f"   Clusters: {best_overall['k']}")
        print(f"   Silhouette Score: {best_overall['silhouette']:.4f}")
        print(f"   Calinski-Harabasz: {best_overall['calinski']:.2f}")
        print(f"   Davies-Bouldin: {best_overall['davies']:.4f}")
        print(f"   Uniqueness Score: {best_overall['uniqueness_score']:.4f}")
        print(f"   Quality: {best_overall['quality']}")
        
        print(f"\n🥈 BEST SILHOUETTE SCORE:")
        print(f"   Data Type: {best_silhouette['data_type']}")
        print(f"   GMM Type: {best_silhouette['gmm_type']}")
        print(f"   Silhouette: {best_silhouette['silhouette']:.4f}")
        print(f"   Clusters: {best_silhouette['k']}")
        
        print(f"\n🥉 BEST BALANCED CLUSTERING:")
        print(f"   Data Type: {best_balanced['data_type']}")
        print(f"   GMM Type: {best_balanced['gmm_type']}")
        print(f"   Clusters: {best_balanced['k']}")
        print(f"   Silhouette: {best_balanced['silhouette']:.4f}")
        
        return {
            'best_overall': best_overall,
            'best_silhouette': best_silhouette,
            'best_balanced': best_balanced,
            'results_df': results_df
        }
    
    def visualize_unique_clusters(self, results_summary):
        """Create visualizations focusing on cluster uniqueness"""
        
        print("\n📊 Creating unique cluster visualizations...")
        
        results_df = results_summary['results_df']
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # 1. Silhouette vs Number of Clusters
        scatter1 = axes[0,0].scatter(results_df['k'], results_df['silhouette'], 
                                    c=results_df['uniqueness_score'], cmap='viridis', s=60, alpha=0.7)
        axes[0,0].set_xlabel('Number of Clusters')
        axes[0,0].set_ylabel('Silhouette Score')
        axes[0,0].set_title('Silhouette Score vs Number of Clusters')
        plt.colorbar(scatter1, ax=axes[0,0], label='Uniqueness Score')
        
        # 2. Data Type Performance
        data_performance = results_df.groupby('data_type')['silhouette'].agg(['max', 'mean']).sort_values('max', ascending=False)
        x_pos = range(len(data_performance))
        bars1 = axes[0,1].bar(x_pos, data_performance['max'], alpha=0.7, label='Max Silhouette')
        bars2 = axes[0,1].bar(x_pos, data_performance['mean'], alpha=0.5, label='Mean Silhouette')
        axes[0,1].set_xlabel('Data Type')
        axes[0,1].set_ylabel('Silhouette Score')
        axes[0,1].set_title('Performance by Data Type')
        axes[0,1].set_xticks(x_pos)
        axes[0,1].set_xticklabels(data_performance.index, rotation=45, ha='right')
        axes[0,1].legend()
        
        # 3. GMM Type Comparison
        gmm_performance = results_df.groupby('gmm_type')['silhouette'].agg(['max', 'mean']).sort_values('max', ascending=False)
        bars3 = axes[0,2].bar(range(len(gmm_performance)), gmm_performance['max'], color=['red', 'green', 'blue'], alpha=0.7)
        axes[0,2].set_xlabel('GMM Type')
        axes[0,2].set_ylabel('Max Silhouette Score')
        axes[0,2].set_title('Performance by GMM Type')
        axes[0,2].set_xticks(range(len(gmm_performance)))
        axes[0,2].set_xticklabels(gmm_performance.index)
        
        # 4. Quality Distribution
        quality_counts = results_df['quality'].value_counts()
        colors = ['red', 'orange', 'yellow', 'lightgreen', 'green'][:len(quality_counts)]
        axes[1,0].pie(quality_counts.values, labels=quality_counts.index, autopct='%1.1f%%', colors=colors)
        axes[1,0].set_title('Distribution of Clustering Quality')
        
        # 5. Uniqueness Score Distribution
        axes[1,1].hist(results_df['uniqueness_score'], bins=20, color='purple', alpha=0.7, edgecolor='black')
        axes[1,1].axvline(results_df['uniqueness_score'].mean(), color='red', linestyle='--', label='Mean')
        axes[1,1].axvline(results_df['uniqueness_score'].quantile(0.9), color='green', linestyle='--', label='90th Percentile')
        axes[1,1].set_xlabel('Uniqueness Score')
        axes[1,1].set_ylabel('Frequency')
        axes[1,1].set_title('Distribution of Uniqueness Scores')
        axes[1,1].legend()
        
        # 6. Top Configurations
        top_10 = results_df.nlargest(10, 'uniqueness_score')
        bars6 = axes[1,2].barh(range(len(top_10)), top_10['uniqueness_score'], color='darkgreen', alpha=0.7)
        axes[1,2].set_ylabel('Configuration Rank')
        axes[1,2].set_xlabel('Uniqueness Score')
        axes[1,2].set_title('Top 10 Unique Clustering Configurations')
        axes[1,2].set_yticks(range(len(top_10)))
        axes[1,2].set_yticklabels([f"{row['data_type'][:10]}\n{row['gmm_type']}" for _, row in top_10.iterrows()])
        
        plt.tight_layout()
        plt.show()
    
    def export_unique_clustering_results(self, results_summary, output_filename=None):
        """Export comprehensive results focusing on unique clusters"""
        
        if output_filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_filename = f'advanced_gmm_unique_clusters_{timestamp}.xlsx'
        
        print(f"\n💾 Exporting unique clustering results to: {output_filename}")
        
        results_df = results_summary['results_df']
        
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            # All results sorted by uniqueness score
            results_df.sort_values('uniqueness_score', ascending=False).to_excel(
                writer, sheet_name='All_Results_by_Uniqueness', index=False)
            
            # Best configurations summary
            best_configs = pd.DataFrame([
                {'Category': 'Best Overall', 'Data_Type': results_summary['best_overall']['data_type'],
                 'GMM_Type': results_summary['best_overall']['gmm_type'], 
                 'Config': results_summary['best_overall']['config'],
                 'K': results_summary['best_overall']['k'],
                 'Silhouette': results_summary['best_overall']['silhouette'],
                 'Uniqueness_Score': results_summary['best_overall']['uniqueness_score']},
                {'Category': 'Best Silhouette', 'Data_Type': results_summary['best_silhouette']['data_type'],
                 'GMM_Type': results_summary['best_silhouette']['gmm_type'],
                 'Config': results_summary['best_silhouette']['config'],
                 'K': results_summary['best_silhouette']['k'],
                 'Silhouette': results_summary['best_silhouette']['silhouette'],
                 'Uniqueness_Score': results_summary['best_silhouette']['uniqueness_score']},
                {'Category': 'Best Balanced', 'Data_Type': results_summary['best_balanced']['data_type'],
                 'GMM_Type': results_summary['best_balanced']['gmm_type'],
                 'Config': results_summary['best_balanced']['config'],
                 'K': results_summary['best_balanced']['k'],
                 'Silhouette': results_summary['best_balanced']['silhouette'],
                 'Uniqueness_Score': results_summary['best_balanced']['uniqueness_score']}
            ])
            best_configs.to_excel(writer, sheet_name='Best_Configurations', index=False)
            
            # Performance by data type
            data_performance = results_df.groupby('data_type').agg({
                'silhouette': ['max', 'mean', 'std'],
                'k': ['mean'],
                'uniqueness_score': ['max', 'mean']
            }).round(4)
            data_performance.to_excel(writer, sheet_name='Performance_by_Data_Type')
            
            # Performance by GMM type
            gmm_performance = results_df.groupby('gmm_type').agg({
                'silhouette': ['max', 'mean', 'std'],
                'k': ['mean'],
                'uniqueness_score': ['max', 'mean']
            }).round(4)
            gmm_performance.to_excel(writer, sheet_name='Performance_by_GMM_Type')
            
            # Top unique configurations (top 20)
            top_unique = results_df.nlargest(20, 'uniqueness_score')[
                ['data_type', 'gmm_type', 'config', 'k', 'silhouette', 'calinski', 'davies', 'uniqueness_score', 'quality']
            ]
            top_unique.to_excel(writer, sheet_name='Top_20_Unique_Clusters', index=False)
        
        print(f"✅ Unique clustering results exported successfully!")
        return output_filename

# Main execution function
def advanced_gmm_unique_clustering(file_path):
    """
    Complete advanced GMM clustering pipeline focused on discovering unique clusters
    """
    
    print("🚀 ADVANCED GMM UNIQUE CLUSTERING SYSTEM")
    print("=" * 50)
    
    # Initialize system
    gmm_system = AdvancedGMMClusteringSystem()
    
    # Load data
    df = pd.read_excel(file_path)
    feature_cols = [col for col in df.columns if col != 'Unique_ID']
    
    print(f"✅ Loaded {len(df)} friends with {len(feature_cols)} features")
    
    # Advanced preprocessing
    preprocessed_data = gmm_system.advanced_data_preprocessing(df, feature_cols)
    
    # Advanced GMM optimization
    results = gmm_system.advanced_gmm_optimization(preprocessed_data)
    
    # Find best unique clusters
    results_summary = gmm_system.find_best_unique_clusters()
    
    if results_summary is None:
        print("❌ No valid clustering results found")
        return None
    
    # Create visualizations
    gmm_system.visualize_unique_clusters(results_summary)
    
    # Export results
    output_file = gmm_system.export_unique_clustering_results(results_summary)
    
    print(f"\n🎉 ADVANCED GMM CLUSTERING COMPLETE!")
    print(f"✅ Tested {len(results)} different configurations")
    print(f"🏆 Best Silhouette Score: {results_summary['best_silhouette']['silhouette']:.4f}")
    print(f"🎯 Best Overall Clusters: {results_summary['best_overall']['k']}")
    print(f"📁 Results saved to: {output_file}")
    
    return gmm_system, results_summary

# Run advanced GMM clustering
if __name__ == "__main__":
    gmm_system, results_summary = advanced_gmm_unique_clustering('./las3.18.xlsx')
    
    if results_summary:
        print(f"\n📋 TOP 5 UNIQUE CLUSTERING CONFIGURATIONS:")
        top_5 = results_summary['results_df'].nlargest(5, 'uniqueness_score')[
            ['data_type', 'gmm_type', 'config', 'k', 'silhouette', 'uniqueness_score', 'quality']
        ]
        print(top_5.to_string(index=False))



🚀 ADVANCED GMM UNIQUE CLUSTERING SYSTEM
✅ Loaded 2425 friends with 10 features
🔧 Advanced data preprocessing...
   ✅ Robust encoding: (2425, 20)


Batches: 100%|██████████| 76/76 [00:54<00:00,  1.40it/s]


   ✅ Enhanced embeddings: (2425, 50)
   ✅ Feature engineered: (2425, 36)
   ✅ Outlier removed: (2089, 10) (removed 336 outliers)
   ✅ Variance filtered: (2425, 10)

🎯 Advanced GMM optimization on 5 data representations...
Data Type            GMM Type              Config   K  Silhouette    Calinski     Davies      Quality
--------------------------------------------------------------------------------------------------------------

--- Processing robust_encoding ---
robust_encoding      Standard         full-kmeans   3       0.138      349.86      2.392         Poor
robust_encoding      Standard         full-kmeans   4       0.126      272.67      2.323         Poor
robust_encoding      Standard         full-kmeans   5       0.091      202.43      2.595         Poor
robust_encoding      Standard         full-kmeans   6       0.071      172.63      3.210         Poor
robust_encoding      Standard         full-kmeans   7       0.073      163.80      3.024         Poor
robust_encoding    